| Column           | النوع   | الشرح                                                            |
| ---------------- | ------- | ---------------------------------------------------------------- |
| id               | int64   | رقم تعريفي فريد لكل عميل (Identifier) — ملوش تأثير تحليلي غالبًا |
| gender           | object  | نوع العميل (Male / Female)                                       |
| SeniorCitizen    | float64 | هل العميل كبير سن؟ (1 = نعم، 0 = لا)                             |
| Partner          | object  | هل العميل لديه شريك/متزوج؟ (Yes / No)                            |
| Dependents       | object  | هل لديه أشخاص يعولهم؟ (Yes / No)                                 |
| tenure           | float64 | عدد الشهور اللي العميل قضاها مع الشركة                           |
| PhoneService     | object  | هل مشترك في خدمة الهاتف؟                                         |
| MultipleLines    | object  | هل عنده أكتر من خط تليفون؟                                       |
| InternetService  | object  | نوع خدمة الإنترنت (DSL / Fiber optic / No)                       |
| OnlineSecurity   | object  | هل مشترك في خدمة الحماية أونلاين؟                                |
| OnlineBackup     | object  | هل لديه خدمة نسخ احتياطي أونلاين؟                                |
| DeviceProtection | object  | هل عنده خدمة حماية أجهزة؟                                        |
| TechSupport      | object  | هل مشترك في الدعم الفني؟                                         |
| StreamingTV      | object  | هل يستخدم خدمة بث التلفزيون؟                                     |
| StreamingMovies  | object  | هل يستخدم خدمة بث الأفلام؟                                       |
| Contract         | object  | نوع التعاقد (Month-to-month / One year / Two year)               |
| PaperlessBilling | object  | هل الفواتير إلكترونية؟                                           |
| PaymentMethod    | object  | طريقة الدفع (Credit Card / Bank Transfer / Electronic check...)  |
| MonthlyCharges   | float64 | قيمة الاشتراك الشهري                                             |
| TotalCharges     | float64 | إجمالي ما دفعه العميل طوال فترة الاشتراك                         |
| Churn            | object  | هل العميل ترك الشركة؟ (Yes / No) ← 🎯 ده ال Target               |


# 1) include used Libraries

In [ ]:
!pip install xgboost lightgbm catboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    roc_auc_score,roc_curve,
    precision_recall_curve,average_precision_score
)
# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.naive_bayes import GaussianNB

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

# 2) Read the Dataset

In [ ]:
df= pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')

In [ ]:
df.head()

In [ ]:
df.shape

# 3) Data Preprocessing

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(8)

In [ ]:
df = df.dropna()

In [ ]:
df = df.drop('id',axis=1)

In [ ]:
df.info()

In [ ]:
df = df.drop_duplicates()

In [ ]:
df['MultipleLines'].unique()

# 4) LabelEncoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Identify categorical columns
categorical_cols = df.select_dtypes(include='object').columns

# Apply Label Encoding
label_encoder = LabelEncoder()
for col in categorical_cols:
    df[col] = label_encoder.fit_transform(df[col])

df.head()

# 5) Visualization

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='Churn', data=df)
plt.title('Distribution of Churn Classes')
plt.xlabel('Churn')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(16, 12))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap of All Features')
plt.show()

In [ ]:
df['tenure'] = df['tenure'].astype(int)
df.info()

In [ ]:
df.reset_index(drop=True, inplace=True)

# 6) Train Test Split

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [ ]:
# Split into Input and Output Elements
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=100, stratify=y)

In [ ]:
df.info()

In [ ]:
# MinMaxScaler
my_scaler = MinMaxScaler()
X_train = my_scaler.fit_transform(X_train)
X_test = my_scaler.transform(X_test)

# 7) Write Down ALL Used Models

In [ ]:
models = {

    # Linear
    #"Logistic Regression": LogisticRegression(max_iter=2000, random_state=1),

    # Distance-based
    #"KNN": KNeighborsClassifier(n_neighbors=5),

    # SVM
    #"SVC": SVC(probability=True, random_state=1),

    # Tree
    "Decision Tree": DecisionTreeClassifier(random_state=1),

    # Naive Bayes
    "Naive Bayes": GaussianNB(),

    # Bagging Based
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, random_state=1),

    # Boosting
    "AdaBoost": AdaBoostClassifier(random_state=1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=1),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=1),

    # External Boosting Libraries
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=1, use_label_encoder=False),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=1),
    "LightGBM": LGBMClassifier(random_state=1)
}

In [ ]:
results = {}

for name, model in tqdm(models.items()):
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred  = model.predict(X_test)

    results[name] = {
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Test Accuracy": accuracy_score(y_test, test_pred),
        "Train Precision": precision_score(y_train, train_pred),
        "Test Precision": precision_score(y_test, test_pred),
        "Train Recall": recall_score(y_train, train_pred),
        "Test Recall": recall_score(y_test, test_pred),
        "Train F1 Score": f1_score(y_train, train_pred),
        "Test F1 Score": f1_score(y_test, test_pred),
        "Train Balanced Accuracy": balanced_accuracy_score(y_train, train_pred),
        "Test Balanced Accuracy": balanced_accuracy_score(y_test, test_pred)

    }

    cm = confusion_matrix(y_test, test_pred)
    print("\nConfusion Matrix:\n", cm)

    plt.figure(figsize=(6,4))
    sns.heatmap(cm,annot=True,fmt="d",cmap='Blues',cbar=False,
                xticklabels=['No','Yes']
                ,yticklabels=['No','Yes'])
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.title('Confusion Matrix')
    plt.show()

    # Full classification report
    print("\nClassification Report:\n")
    print(classification_report(y_test, test_pred,target_names=['No','Yes'],zero_division=0))

    y_pred_prob = model.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    print("ROC-AUC:", roc_auc)

    # ROC curve
    fpr, tpr, roc_thresholds = roc_curve(y_test, y_pred_prob)
    plt.figure()
    plt.plot(fpr, tpr, label='ROC curve (area = {:.2f})'.format(roc_auc))
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic')
    plt.legend(loc="lower right")
    plt.show()

    pr_auc = average_precision_score(y_test, y_pred_prob)
    print("PR AUC:", pr_auc)

    precision_vals, recall_vals, pr_thresholds = precision_recall_curve(y_test, y_pred_prob)
    plt.figure()
    plt.plot(recall_vals, precision_vals, label='Precision-Recall curve')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend(loc="lower left")
    plt.show()

# 8) Print the Results

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by="Test F1 Score", ascending=False)

In [ ]:
results_df

# 9) Apply Catboost Again

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# 1. قراءة البيانات (تأكد من المسارات)
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

# 2. تجهيز الـ Features والـ Target
X = train[train.columns[1:-1]]
y = (train.Churn == 'Yes').astype(int)
X_test_final = test[test.columns[1:]]

# 3. حل مشكلة الـ CATS (تحديد الأعمدة النصية)
CATS = X.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical Features found: {CATS}")

# 4. إعدادات الـ Cross-Validation
n_splits = 12
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
oof_pred = np.zeros(len(X))
test_pred = np.zeros(len(X_test_final))

# 5. الـ Training Loop
for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    cb = CatBoostClassifier(
        n_estimators=8000, 
        learning_rate=0.04,
        eval_metric='AUC', 
        auto_class_weights='Balanced',
        random_state=42,
        early_stopping_rounds=100,
        task_type='GPU', # تأكد إن عندك GPU شغال، لو مفيش غيرها لـ 'CPU'
        cat_features=CATS,
        verbose=500 
    )

    cb.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

    val_pred = cb.predict_proba(X_val)[:, 1]
    oof_pred[val_index] = val_pred   
    test_pred += cb.predict_proba(X_test_final)[:, 1] / n_splits

    print(f"Fold {fold+1} AUC: {roc_auc_score(y_val, val_pred):.5f}")

print(f"\nFinal CV AUC: {roc_auc_score(y, oof_pred):.5f}")

In [ ]:
# تأكد إن اسم العمود هو اللي مطلوب في المسابقة (غالبا بيكون Churn أو Target)
target_column_name = 'Churn' 

submission = pd.DataFrame({
    "id": test["id"],
    target_column_name: test_pred
})

# حفظ الملف
submission.to_csv("submission.csv", index=False)

print("✅ Submission file saved successfully!")
# عرض أول 5 صفوف للتأكد من الشكل النهائي
print(submission.head())